In [ ]:
import os
from typing import List, Dict, Any
import pandas as pd
from langchain_core.documents import Document # Document structure in LangChain
from langchain_text_splitters.character import(
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)

from langchain_text_splitters import TokenTextSplitter

In [9]:
doc = Document(
    page_content="This is the first review about langchain document structure",
    metadata={
        "source":"me",
        "time":"18:36"
    }
)

print(f"Content:{doc.page_content}")
print(f"Metadata:{doc.metadata}")

Content:This is the first review about langchain document structure
Metadata:{'source': 'me', 'time': '18:36'}


In [10]:
type(doc)

langchain_core.documents.base.Document

# Why metadata matters:
It is very crucial for:
1. Filtering search results
2. Tracking document sources
3. Providing context in response
4. Debugging and auditing

In [ ]:
# create a file to store text files
os.makedirs("data/text_files",exist_ok=True)

In [ ]:
sample_texts = {
    "data/text_files/comment1.txt":"As a single 28yr old male that struggles with depression & manic anxiety... Watching this video on my couch alone at midnight was an experience I didn't know I needed.... Thank you, that was beautiful"
}

In [29]:
sample_texts2 = {
    "data/text_files/comment2.txt":"""Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """
}

In [30]:
for path, text in sample_texts2.items():
    with open(path,'w',encoding='utf-8') as f:
        f.write(text)

# TextLoader : read a single file

In [31]:
from langchain_community.document_loaders import TextLoader

# Loading a single text file
loader = TextLoader("data/text_files/comment2.txt",encoding='utf-8')

documents = loader.load()
print(type(documents))
print(f"Content: {documents[0].page_content}")
print(f"Metadata: {documents[0].metadata}")

<class 'list'>
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


    
Metadata: {'source': 'data/text_files/comment2.txt'}


# DirectoryLoader: load multiple files

In [33]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)

documents = dir_loader.load()

print(f"Loaded {len(documents)} documents")

for i, doc in enumerate(documents):
    print(f"Document {i+1}: ")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")

100%|██████████| 2/2 [00:00<00:00, 783.18it/s]

Loaded 2 documents
Document 1: 
Content: As a single 28yr old male that struggles with depression & manic anxiety... Watching this video on my couch alone at midnight was an experience I didn't know I needed.... Thank you, that was beautiful
Metadata: {'source': 'data\\text_files\\comment1.txt'}
Document 2: 
Content: Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


    
Metadata: {'source': 'data\\text_files\\comment2.txt'}


# Text Splitting Strategies

In [46]:
text = documents[1].page_content
text

'Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '

In [ ]:
# Method 1: Character-based splitting

char_splitter = CharacterTextSplitter(
    separator=' ',
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

char_chunks = char_splitter.split_text(text)

print(f"Created {len(char_chunks)} chunks")
for i in range(len(char_chunks)):
    print(f"Chunk {i+1}")
    print(char_chunks[i])
    print()

Created 3 chunks
Chunk 1
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing
Chunk 2
on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning:
Chunk 3
Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


In [45]:
# Method 2: Recursive Character Splitting

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n','\n',' ',''],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)

print(f"Created {len(recursive_chunks)} chunks")
for i in range(len(recursive_chunks)):
    print(f"Chunk {i+1}")
    print(recursive_chunks[i])
    print()


Created 6 chunks
Chunk 1
Machine Learning Basics

Chunk 2
Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs

Chunk 3
that can access data and use it to learn for themselves.

Chunk 4
Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data

Chunk 5
3. Reinforcement Learning: Learning through rewards and penalties

Chunk 6
Applications include image recognition, speech processing, and recommendation systems



In [49]:
print(text)

Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


    


In [48]:
# Method 3: Token-based splitting

token_splitter = TokenTextSplitter(
    chunk_size=50, # size in token
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)

print(f"Created {len(token_chunks)} chunks")
for i in range(len(token_chunks)):
    print(f"Chunk {i+1}")
    print(token_chunks[i])
    print()

Created 3 chunks
Chunk 1
Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types

Chunk 2
 use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards

Chunk 3

3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


    

